<font color="#8E44AD" size='6px'><b>Image Segmentation</b></font>

<font color="#8E44AD" size='4px'><b>The Pipeline Architecture</b></font>

The system operates in a dense segmentation pipeline designed to identify, separate, and label every distinct object in the scene with pixel-level accuracy. It integrates Vision-Language Models for discovery with **SAM 3** for high-fidelity masking .

<font color="#D35400" size='4px'><b>Stage I: Scene Understanding & Class Discovery</b></font>
Before segmentation begins, the system must "read" the image to know what objects exist.

**Automatic Discovery:** The image is analyzed by a Vision-Language Model to generate a list of candidate objects.
* <font color="#884EA0"><i>LLM Mode:</i></font> Uses **Async Parallel Processing** to analyze multiple crops of the image simultaneously, detecting complex or subtle objects (e.g., "vintage lamp", "specific brand logo").
* <font color="#884EA0"><i>BLIP Mode:</i></font> Uses local captioning models to find standard nouns (e.g., "chair", "table").

**NLP Refinement:** The raw text descriptions are parsed using **SpaCy**. Abstract concepts (e.g., "atmosphere", "view") are filtered out, and synonyms are merged to create a clean list of physical target nouns.

<font color="#D35400" size='4px'><b>Stage II: SAM 3 Instance Segmentation</b></font>
The clean noun list serves as the input prompt for the segmentation engine.

**Text-Prompted Inference:** Each discovered noun (e.g., "cat") is fed into the **SAM 3 (Segment Anything Model 3)** as a text prompt.

**Mask Generation:** SAM 3 predicts binary **Spatial Masks** for every instance of that object. Unlike semantic segmentation (which groups all "cars" into one blob), SAM 3 distinguishes between "Car A" and "Car B."

<font color="#D35400" size='4px'><b>Stage III: Post-Processing & Filtering</b></font>
Raw model outputs can contain noise or overlapping duplicates.

**Confidence Thresholding:** Masks with low confidence scores (below `SAM3_CONF_IMAGE_THRESHOLD`) are discarded to ensure quality.

**Non-Maximum Suppression (NMS):** If two masks overlap significantly (IoU > Threshold), the system keeps only the higher-confidence one, preventing double-counting of the same object.

<font color="#D35400" size='4px'><b>Stage IV: Visualization & Map Assembly</b></font>
The final step constructs the interpretable output maps.

**Harmonious Coloring:** Each object class is assigned a unique, high-contrast color using a hue-based generator (`generate_harmonious_colors`) to ensure readability.

**Map Generation:**
* <font color="#884EA0"><i>Instance Map:</i></font> A color-coded map where every pixel belongs to a specific object instance.
* <font color="#884EA0"><i>Legend:</i></font> A dynamically generated legend mapping colors to class names (e.g., "Red = Person", "Blue = Bicycle").

<font color="#2E86C1" size='6px'>Environment Initialization & Repository Setup</font>

In [ ]:
COLAB = False

In [ ]:
import os
import sys

if COLAB:
    # Clone Repository 
    if not os.path.exists("sam3-toolkit"):
        print("--- Cloning repository...")
        !git clone https://github.com/MrKGhasemi/sam3-toolkit.git
    else:
        print("--- Repository already exists.")

    # Auto-Detect Paths 
    repo_root = os.path.abspath("sam3-toolkit")
    sam3_install_path = None
    practical_path = None

    for root, dirs, files in os.walk(repo_root):
        if ("sam3" in dirs or "sam3_main" in root or root == repo_root):
            if sam3_install_path is None or len(root) < len(sam3_install_path):
                sam3_install_path = root

        if "image_segmentation.py" in files:
            practical_path = root

    if not sam3_install_path or not practical_path:
        raise FileNotFoundError(
            "     --- Critical paths (sam3 or practical) not found.")

    print(f"Found SAM 3 Library at: {sam3_install_path}")
    print(f"Found Project Code at: {practical_path}")

    # Install Dependencies 
    print("--- Installing Python packages...")
    !{sys.executable} -m pip install -q opencv-python matplotlib scikit-learn transformers spacy openai gdown mega.py huggingface_hub
    !{sys.executable} -m pip install -q einops decord pycocotools

    # Install SAM 3 
    print("--- Installing SAM 3 Library...")
    !{sys.executable} -m pip install -e "{sam3_install_path}"

    # Configure Working Directory 
    os.chdir(practical_path)
    if practical_path not in sys.path:
        sys.path.insert(0, practical_path)
    print(f"--- Working directory set to: {os.getcwd()}")

    # Download Model Weights
    weights_dir = "weights"
    weights_path = os.path.join(weights_dir, "sam3.pt")
    os.makedirs(weights_dir, exist_ok=True)

    # If the model is Private/Gated, you MUST provide a token.
    # Get it here: https://huggingface.co/settings/tokens
    HF_REPO_ID = "facebook/sam3" 
    HF_FILENAME = "sam3.pt"               
    HF_TOKEN = "HF_TOKEN"  # Replace with your actual token or set to None

    if not os.path.exists(weights_path):
        print(f"--- Starting download...")

        try:
            from huggingface_hub import hf_hub_download
            print("--- Connecting to Hugging Face Hub...")
            downloaded_file = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=HF_FILENAME,
                local_dir=weights_dir,
                token=HF_TOKEN,
                local_dir_use_symlinks=False  # Ensure we get the actual file, not a symlink
            )
            if os.path.basename(downloaded_file) != "sam3.pt":
                os.rename(downloaded_file, weights_path)

            print("--- Download attempt finished.")

        except Exception as e:
            print(f"      --- Error downloading: {e}")
            print("           Tip: If using Hugging Face private repo, ensure HF_TOKEN is set.")

    else:
        print("--- Weights file already exists.")

    # Verify File Integrity 
    if os.path.exists(weights_path):
        final_size = os.path.getsize(weights_path) / (1024 * 1024)
        print(f"--- Final File Size: {final_size:.2f} MB")
        if final_size < 2000:
            print(
                "      --- WARNING: File seems too small (<2GB). It might be corrupt or a placeholder.")
    else:
        print("      --- Error: File not found after download.")

    # Download SpaCy Model 
    print("--- Checking SpaCy model...")
    !{sys.executable} -m spacy download en_core_web_lg

    # Verify Import 
    print("\n--- Verifying imports...")
    repo_root = os.path.abspath("/sam3-toolkit")

    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
        print(f"--- Added '{repo_root}' to sys.path")

    practical_dir = os.path.join(repo_root, "practical")
    if practical_dir not in sys.path:
        sys.path.insert(0, practical_dir)
        print(f"--- Added '{practical_dir}' to sys.path")

    # Verify all dependencies
    print("\n--- Retrying imports...")
    try:
        import sam3
        print("--- 'sam3' library loaded!")
    except ImportError as e:
        print(f"      --- Failed to load sam3: {e}")
        print("       --- Attempting hard install fix...")
        !pip install "{repo_root}"
        import sam3

    try:
        import models
        print("--- 'models.py' loaded!")
    except ImportError as e:
        print(f"      --- Failed to load models: {e}")

    !pip uninstall numpy -y
    !{sys.executable} -m pip install "numpy<2.0"

    # Check GPU
    import torch
    print(f"--- GPU Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"--- GPU Name: {torch.cuda.get_device_name(0)}")

    if os.path.exists(practical_dir):
        os.chdir(practical_dir)
        print(f"--- Current Directory: {os.getcwd()}")


---

<font color="#117A65" size='6px'>Loading Core Modules & Dependencies</font>

<font color="#D68910" size='4px'>*image_segmentation:*</font> Contains the main pipeline for generate instance/semantic segmentation.

<font color="#D68910" size='4px'>*visualization:*</font> Handles the drawing of bounding boxes, masks, and legends.



In [ ]:
import torch
import os
import sys

root = os.path.abspath(os.path.join(os.getcwd(), ".."))

sys.path.append(root)

if os.path.basename(os.getcwd()) == "ipynb":
    os.chdir(root)

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import visualization
import image_segmentation

In [ ]:
if COLAB:
    IMAGE_FILENAME = "images/sa_717.jpg"
    IMAGE_PATH = os.path.join(repo_root, "images", IMAGE_FILENAME)

    if not os.path.exists(IMAGE_PATH):
        alt_path = os.path.join(repo_root, IMAGE_FILENAME)
        if os.path.exists(alt_path):
            VIDEO_PATH = alt_path
        else:
            print(f"      --- Warning: Could not find {IMAGE_FILENAME} in {os.path.dirname(IMAGE_PATH)}")
else:        
    IMAGE_PATH = "images/sa_717.jpg"

```Python
# From configs.py 
LLM_CAPTION_MODEL_NAME = "gpt-5-chat"
SAM3_CONF_IMAGE_THRESHOLD = 0.3
```

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


class MockArgs:
    def __init__(self, mode, output_dir, api_key=None, base_url=None):
        self.mode = mode
        self.output = output_dir
        self.api_key = api_key
        self.base_url = base_url


output_directory = "notebook_outputs"
os.makedirs(output_directory, exist_ok=True)

args = MockArgs(mode="blip", output_dir=output_directory)

# for LLM mode:
# args = MockArgs(
#     mode="llm",
#     output_dir=output_directory,
#     api_key="api_key",
#     base_url="base_url"
# )

---

<font color="#D35400" size='6px'>Execution1: BLIP Automatic Detection</font>

This cell runs the Standard Automatic Mode. By using <font color="#D35400" size='4px'>args.mode='blip'</font>, employ a locally hosted Vision-Language Model to "look" at image and figure out what's inside without needing external APIs.

The function <font color="#D35400"><i>get_classes_blip</i></font> orchestrates a smart scanning process:

<font color="#D35400" size='4px'><i>Smart Cropping:</i></font> To ensure don't miss small details, the image is sliced into multiple strategic crops (zoom-ins) using <font color="#D35400"><i>visualization.get_image_crops</i></font>. This allows the AI to focus on specific regions rather than just the cluttered whole.

<font color="#D35400" size='4px'><i>Local Captioning:</i></font> Runs the <font color="#c5885f"><i>BlipForConditionalGeneration</i></font> model on every single crop. It generates a descriptive sentence for each section,  creating a textual map of the image content.

<font color="#D35400" size='4px'><i>Noun Extraction:</i></font> Finally, all these captions are stitched together. then uses a standard NLP library (SpaCy) to parse this text and extract the specific nouns (e.g., "car", "dog"), creating a clean target list for SAM3 to segment.

```python
def process_image(image_path, args, save_viz=False):
    # ...
    # 1. NLP Phase: Generate Classes
    if args.mode == "blip":
        noun_phrases, _ = class_generators.get_classes_blip(...)
    
    # 2. Model Phase: Segment with SAM 3
    sam3_model, sam3_processor = models.load_sam3_model(...)
```    

In [ ]:
output_information = image_segmentation.process_image(IMAGE_PATH, args, save_viz=False)

<font color="#9584" size='6px'>Visualizing the Results</font>


In [ ]:
if output_information["masks"]:
    print("Displaying results in notebook...")
    visualization.show_save_visualizations(output_information, figsize=(45, 30))
else:
    print("No masks were generated for this image.")

---

<font color="#3c983c" size='6px'>Execution2: LLM-Enhanced Reasoning</font>

This cell activates the LLM Mode. By simply setting <font color="#3c983c" size='4px'>args.mode='llm'</font>, bypass the standard local models and instead use a Multimodal LLM to get a much deeper understanding of the scene.

This function runs on a fast Async Pipeline:

<font color="#CB4335" size='4px'><i>Asyncio Parallelism:</i></font> unlike BLIP (which works sequentially), the function <font color="#CB4335"><i>get_classes_llm</i></font> chops the image into multiple detailed crops using <font color="#CB4335" size='4px'><i>visualization.get_image_crops*</i></font>. It then sends API requests for all those crops at the exact same time, analyzing the whole image in parallel rather than waiting.

<font color="#CB4335" size='4px'><i>Visual Reasoning:</i></font> Instead of a generic "caption", instruct the model to focus strictly on describing physical objects. This allows the system to spot subtle or complex items that standard BLIP models usually miss.

<font color="#CB4335" size='4px'><i>Intelligent Parsing:</i></font> The raw descriptions from the vision model are collected and sent to a second "Parser" LLM. This step, managed by <font color="#CB4335"><i>utils.search_engine_clean_nouns</i></font>, cleans up the text and extracts a neat list of target nouns that act as precise prompts for SAM3.

```python
# The Async logic that powers LLM mode
async def run_parallel_captions():
    # ...
    # Process multiple crops in parallel for speed
    results = await asyncio.gather(*tasks)
    return [r for r in results if r]
```    

```python
# ...Used by both Image Segmentation...
# 1. Prepare Prompts (Class Names)
# target_nouns is a list like ['car', 'person', 'tree']
prompt_list = [[n] for n in target_nouns] 

# 2. Batch Prediction (Single Image, Multiple Prompts)
# We pass the same image for every prompt
images = [raw_image] * len(prompt_list)

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
    # 3. Query SAM 3 Model
    outputs = sam3_processor(
        images=images,
        text_prompts=prompt_list,  # Prompts: [["car"], ["person"]]
        return_tensors="pt",
        padding=True
    ).to(device)

    # 4. Parse Raw Output
    # SAM 3 returns masks, IoU scores, and object scores
    results = sam3_processor.post_process_masks(
        outputs, 
        target_sizes=[(height, width)] * len(prompt_list),
        conf_threshold=configs.SAM3_CONF_IMAGE_THRESHOLD
    )

# 5. Aggregate Results
final_masks = []
for i, res in enumerate(results):
    # For each prompt, extract valid masks
    masks = res["masks"]  # Boolean Tensor [N, H, W]
    scores = res["scores"]
    
    for m, s in zip(masks, scores):
        if s > threshold:
            final_masks.append({
                "segmentation": m.cpu().numpy(),
                "class_name": target_nouns[i],
                "score": float(s)
            })
```

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


class MockArgs:
    def __init__(self, mode, output_dir, api_key=None, base_url=None):
        self.mode = mode
        self.output = output_dir
        self.api_key = api_key
        self.base_url = base_url


output_directory = "notebook_outputs"
os.makedirs(output_directory, exist_ok=True)

# for LLM mode:
args = MockArgs(
    mode="llm",
    output_dir=output_directory,
    api_key="api_key",
    base_url="base_url"
)

In [ ]:
output_information = image_segmentation.process_image(IMAGE_PATH, args)

<font color="#9584" size='6px'>Visualizing the Results</font>


In [ ]:
if output_information["masks"]:
    print("Displaying results in notebook...")
    visualization.show_save_visualizations(output_information, figsize=(45, 30))
else:
    print("No masks were generated for this image.")